# 02 — Feature Engineering with Unity Catalog

The [Feature Engineering in Databricks Unity Catalog](https://docs.databricks.com/en/machine-learning/feature-store/uc/feature-tables-uc.html) allows you to create a centralized repository of features. These features can be used to train & call your ML models. By saving features as feature engineering tables in Unity Catalog, you will be able to:

- **Share features** across your organization
- **Increase discoverability** — browse features in Catalog Explorer
- **Ensure consistency** — the same feature computation code is used for model training and inference
- **Track lineage** — Unity Catalog tracks which models use which features

### This notebook

1. Explore the raw sensor data (quick analysis)
2. Compute windowed time-series features for cycle detection
3. Save them as a **Feature Table** in Unity Catalog with `fe.create_table()`
4. Inspect the table with `fe.get_table()`

In [ ]:
%pip install databricks-feature-engineering
dbutils.library.restartPython()

In [ ]:
%run ./00_Config

In [ ]:
# DBTITLE 1,Review the raw sensor data
display(spark.table(RAW_SENSOR_TABLE))

## Quick Data Analysis

Before building features, let's explore the data. We use the data profiler and a quick visualization to understand the distributions and identify potential signal for cycle detection.

In [ ]:
import matplotlib.pyplot as plt

sample_pdf = (
    spark.table(RAW_SENSOR_TABLE)
    .sample(0.05)
    .select("pressure_bar", "temperature_c", "flow_rate", "cycle_status")
    .toPandas()
)

cols = ["pressure_bar", "temperature_c", "flow_rate"]
fig, axes = plt.subplots(1, len(cols), figsize=(16, 4))
for ax, col in zip(axes, cols):
    for label, color in [(1, "#ff7f0e"), (0, "#1f77b4")]:
        ax.hist(sample_pdf.loc[sample_pdf["cycle_status"] == label, col],
                bins=50, alpha=0.5, density=True, label=f"cycle_status={label}", color=color)
    ax.set_xlabel(col)
    ax.set_ylabel("density")
    ax.legend()
fig.suptitle("Sensor Distributions by Cycle Status")
plt.tight_layout()
plt.show()

## 1. Compute the Features

We create windowed time-series features that capture **local signal behavior** around each data point. These are the features our model will use to decide: *is this point part of a brake cycle?*

**Important**: We will **drop the label** (`cycle_status`) from the feature table. Labels should not be part of the features to avoid leaking results to our models. The label will be joined back during training via `FeatureLookup`.

This transformation would typically be part of a job used to refresh features — the same code runs for training and inference.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

raw_df = spark.table(RAW_SENSOR_TABLE)

# Ordered windows per test run
w10 = Window.partitionBy("test_run_id").orderBy("timestamp").rowsBetween(-9, 0)
w50 = Window.partitionBy("test_run_id").orderBy("timestamp").rowsBetween(-49, 0)
w1  = Window.partitionBy("test_run_id").orderBy("timestamp")

features_df = (
    raw_df
    # --- Pressure features ---
    .withColumn("pressure_rolling_mean_10", F.avg("pressure_bar").over(w10))
    .withColumn("pressure_rolling_std_10",  F.stddev("pressure_bar").over(w10))
    .withColumn("pressure_rolling_mean_50", F.avg("pressure_bar").over(w50))
    .withColumn("pressure_rolling_std_50",  F.stddev("pressure_bar").over(w50))
    .withColumn("pressure_min_10",          F.min("pressure_bar").over(w10))
    .withColumn("pressure_max_10",          F.max("pressure_bar").over(w10))
    .withColumn("pressure_range_10",        F.col("pressure_max_10") - F.col("pressure_min_10"))

    # --- Gradient features ---
    .withColumn("pressure_lag_1",  F.lag("pressure_bar", 1).over(w1))
    .withColumn("pressure_gradient", F.col("pressure_bar") - F.col("pressure_lag_1"))
    .withColumn("pressure_gradient_abs", F.abs(F.col("pressure_gradient")))

    # --- Valve state features ---
    .withColumn("valve_lag_1", F.lag("valve_state", 1).over(w1))
    .withColumn("valve_state_change",
                F.when(F.col("valve_state") != F.col("valve_lag_1"), 1).otherwise(0))
    .withColumn("valve_switches_10", F.sum("valve_state_change").over(w10))

    # --- Temperature and flow features ---
    .withColumn("temp_lag_1", F.lag("temperature_c", 1).over(w1))
    .withColumn("temp_gradient", F.col("temperature_c") - F.col("temp_lag_1"))
    .withColumn("flow_rate_rolling_mean_10", F.avg("flow_rate").over(w10))

    # --- Cleanup: drop intermediate lag columns and the label ---
    .drop("pressure_lag_1", "valve_lag_1", "temp_lag_1", "pressure_min_10", "pressure_max_10")
    .drop("cycle_status")  # Labels should NOT be in the feature table
    .fillna(0.0)
)

print(f"Feature columns: {len(features_df.columns)}")
display(features_df.limit(5))

## 2. Save the Feature Table

We use `fe.create_table()` to register the features as a **Feature Table** in Unity Catalog.

We need to give it:
- A **name** (fully qualified Unity Catalog path)
- **Primary keys** for lookup (`test_run_id` + `timestamp`)
- **Timestamp keys** — enables point-in-time correct lookups (preventing data leakage in time-series)

Under the hood, this creates a Delta Table with the primary key constraints registered in Unity Catalog.

In [ ]:
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup

fe = FeatureEngineeringClient(model_registry_uri="databricks-uc")

spark.sql(f"DROP TABLE IF EXISTS {FEATURE_TABLE}")

fe.create_table(
    name=FEATURE_TABLE,
    primary_keys=["test_run_id", "timestamp"],
    timestamp_keys="timestamp",
    df=features_df,
    description="Windowed time-series features for brake cycle detection. "
                "Computed from raw testbench sensor data (pressure, valve state, temperature, flow rate).",
)

### Our Feature Table is now ready!

We can explore the created feature engineering table using the **Unity Catalog Explorer**. Navigate to `{CATALOG}.{SCHEMA}.cycle_detection_features` in your workspace and browse the Features tab.

We can also use `fe.get_table()` to inspect the metadata programmatically:

In [ ]:
ft_info = fe.get_table(name=FEATURE_TABLE)
print(f"Feature Table: {ft_info.name}")
print(f"Description:   {ft_info.description}")
print(f"Primary keys:  {ft_info.primary_keys}")
print(f"Features:      {ft_info.features}")

## 3. Feature Distribution Check

Quick sanity check: do the computed features show different distributions for cycle vs non-cycle points? (We join back the label just for this analysis.)

In [ ]:
features_with_label = (
    fe.read_table(name=FEATURE_TABLE)
    .join(
        spark.table(RAW_SENSOR_TABLE).select("test_run_id", "timestamp", "cycle_status"),
        on=["test_run_id", "timestamp"],
    )
)

summary = (
    features_with_label
    .groupBy("cycle_status")
    .agg(
        F.mean("pressure_rolling_mean_10").alias("avg_pressure_mean_10"),
        F.mean("pressure_rolling_std_10").alias("avg_pressure_std_10"),
        F.mean("pressure_gradient_abs").alias("avg_gradient_abs"),
        F.mean("valve_switches_10").alias("avg_valve_switches_10"),
        F.mean("flow_rate_rolling_mean_10").alias("avg_flow_rate_10"),
        F.count("*").alias("count"),
    )
)

display(summary)

## Summary

We created a **Feature Table** in Unity Catalog with windowed time-series features for cycle detection.

Key takeaways:
- Labels (`cycle_status`) are **not** stored in the feature table — they'll be joined at training time via `FeatureLookup`
- `timestamp_keys` enables point-in-time correct lookups, preventing data leakage
- The table is browsable in Catalog Explorer and reusable by any team/model

**Next**: Open `03_Model_Training_MLflow` to train a cycle detection model using these features.